# MedTriage OCR — Medical Text Recognition Training

Fine-tunes PaddleOCR PP-OCRv3 recognition model on medical vocabulary.

**What this does:** Teaches the OCR to read ward sheet text (patient names, diagnoses, medications, bed numbers) with high accuracy.

**Runtime:** Select **GPU** → T4 (free tier). Training takes ~2-4 hours.

**Privacy:** Only synthetic training images are uploaded. No real patient data.

## Step 1: Install Dependencies

In [ ]:
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install -q paddleocr scikit-image lmdb
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

import paddle
print(f'PaddlePaddle {paddle.__version__}, GPU: {paddle.is_compiled_with_cuda()}')
print(f'GPU device: {paddle.device.get_device()}')

## Step 2: Upload Training Data

Upload the ZIP file created by the cell below on your local machine:
```
cd MedTriage/med_ocr
python -c "import shutil; shutil.make_archive('training_upload', 'zip', 'training_data')"
```
Then upload `training_upload.zip` when prompted.

In [ ]:
from google.colab import files
import os, zipfile

# Upload training data ZIP
print('Upload training_upload.zip from your med_ocr/ folder...')
uploaded = files.upload()

# Extract
os.makedirs('training_data', exist_ok=True)
with zipfile.ZipFile('training_upload.zip', 'r') as z:
    z.extractall('training_data')

# Verify
train_list = 'training_data/train_list.txt'
val_list = 'training_data/val_list.txt'
dict_path = 'training_data/med_dict.txt'

with open(train_list) as f: train_count = sum(1 for _ in f)
with open(val_list) as f: val_count = sum(1 for _ in f)
with open(dict_path) as f: dict_size = sum(1 for _ in f)

print(f'\nTraining samples: {train_count}')
print(f'Validation samples: {val_count}')
print(f'Dictionary: {dict_size} characters')
print(f'Images: {len(os.listdir("training_data/images"))}')
print('\nData loaded successfully!')

## Step 3: Download Pretrained Model

In [ ]:
import urllib.request, tarfile

model_url = 'https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar'
model_tar = 'en_PP-OCRv3_rec_train.tar'

if not os.path.exists('en_PP-OCRv3_rec_train'):
    print('Downloading PP-OCRv3 English recognition model...')
    urllib.request.urlretrieve(model_url, model_tar)
    with tarfile.open(model_tar, 'r') as tar:
        tar.extractall('.')
    os.remove(model_tar)
    print('Model downloaded.')
else:
    print('Model already exists.')

## Step 4: Write Training Config

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
EVAL_EVERY = 500

config = f"""Global:
  debug: false
  use_gpu: true
  epoch_num: {EPOCHS}
  log_smooth_window: 20
  print_batch_step: 50
  save_model_dir: ./trained_model
  save_epoch_step: 5
  eval_batch_step: [0, {EVAL_EVERY}]
  cal_metric_during_train: true
  pretrained_model: ./en_PP-OCRv3_rec_train/best_accuracy
  checkpoints:
  use_visualdl: false
  character_dict_path: ./training_data/med_dict.txt
  character_type: en
  max_text_length: 80
  infer_mode: false
  use_space_char: true
  save_res_path: ./trained_model/rec_results.txt

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 5
  regularizer:
    name: L2
    factor: 0.00001

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: true
          Head:
            fc_decay: 0.00001
      - SARHead:
          enc_dim: 512
          max_text_length: 80

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - SARLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: SimpleDataSet
    data_dir: ./training_data
    label_file_list:
      - ./training_data/train_list.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecAug:
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: true
    batch_size_per_card: {BATCH_SIZE}
    drop_last: true
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./training_data
    label_file_list:
      - ./training_data/val_list.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: false
    batch_size_per_card: {BATCH_SIZE}
    drop_last: false
    num_workers: 4
"""

with open('med_rec_config.yml', 'w') as f:
    f.write(config)

iters_per_epoch = train_count // BATCH_SIZE
total_iters = iters_per_epoch * EPOCHS
print(f'Config written: {EPOCHS} epochs, batch {BATCH_SIZE}')
print(f'{iters_per_epoch} iters/epoch x {EPOCHS} = {total_iters} total iterations')
print(f'Estimated time on T4 GPU: ~2-4 hours')

## Step 5: Train! (GPU accelerated)

In [ ]:
!cd PaddleOCR && python tools/train.py -c ../med_rec_config.yml

## Step 6: Check Best Accuracy

In [ ]:
import pickle

states_path = 'trained_model/best_accuracy.states'
if os.path.exists(states_path):
    with open(states_path, 'rb') as f:
        states = pickle.load(f)
    best = states['best_model_dict']
    print(f"Best Accuracy: {best['acc']:.1%}")
    print(f"Norm Edit Distance: {best['norm_edit_dis']:.3f}")
    print(f"Best Epoch: {best['best_epoch']}")
    print(f"FPS: {best['fps']:.1f}")
else:
    print('Training still in progress...')

# Show training curve
!grep 'best metric' trained_model/train.log | tail -10

## Step 7: Export to ONNX (for browser deployment)

In [ ]:
# Export Paddle inference model
!cd PaddleOCR && python tools/export_model.py \
    -c ../med_rec_config.yml \
    -o Global.pretrained_model=../trained_model/best_accuracy \
    -o Global.save_inference_dir=../trained_model/inference

print('\nInference model exported.')
!ls -lh trained_model/inference/

In [ ]:
# Convert to ONNX
!pip install -q paddle2onnx
!paddle2onnx \
    --model_dir trained_model/inference \
    --model_filename inference.pdmodel \
    --params_filename inference.pdiparams \
    --save_file trained_model/med_rec.onnx \
    --opset_version 14 \
    --enable_onnx_checker True

print('\nONNX model exported!')
!ls -lh trained_model/med_rec.onnx

## Step 8: Download Trained Model

In [ ]:
import shutil

# Package everything for download
download_files = {
    'trained_model/med_rec.onnx': 'The ONNX model for browser deployment',
    'training_data/med_dict.txt': 'Character dictionary (must match model)',
}

shutil.make_archive('medtriage_trained_model', 'zip', '.', 'trained_model')

print('Files ready for download:')
for f, desc in download_files.items():
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f'  {f} ({size:.1f} MB) — {desc}')

print('\nDownloading ZIP...')
files.download('medtriage_trained_model.zip')
print('\nDone! Copy med_rec.onnx to MedTriage/public/models/ocr/latin-rec.onnx')
print('Copy med_dict.txt to MedTriage/public/models/ocr/latin-dict.txt')
print('Then: npm run build && npx firebase deploy --only hosting')